In [61]:
import pandas as pd
import numpy as np
import json
from datetime import datetime

import os
api_key=os.getenv("api_key")
from groq import Groq
client=Groq(api_key=api_key)

In [62]:
from langchain_community.utilities.sql_database import SQLDatabase
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_classic.vectorstores import Chroma
from langchain_community.agent_toolkits import create_sql_agent
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.tools import tool
from pathlib import Path
project_root=Path.cwd().parent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_groq import ChatGroq

In [80]:
# For Api Rate limiting
import logging
from tenacity import (
     retry,  # Decorator that wraps a function with retry logic
    stop_after_attempt,  # Stop after N total attempts
    wait_exponential,  # Wait 1s, 2s, 4s, 8s between retries
    before_sleep_log, 

)


logging.basicConfig(
    level=logging.INFO,
    filename=project_root/"log"/"rag_tool.log",
    filemode="w",

)

logger = logging.getLogger(__name__)
attempt_counter={"n":0}


In [79]:
log_dir = project_root / "log"
log_dir.mkdir(parents=True, exist_ok=True)

# 2. Clear out any existing handlers so Python doesn't write to the old file
for handler in logging.root.handlers[:]:
  logging.root.removeHandler(handler)

# 3. Re-initialize logging fresh for this specific notebook/file
logging.basicConfig(
    level=logging.INFO,
    filename=log_dir / "rag_tool.log",  # Or whatever specific log filename you want here
    filemode="w",
)

logger = logging.getLogger(__name__)


In [64]:
# Initialise the embedding model
embedding_model = SentenceTransformerEmbeddings(model_name='thenlper/gte-large')

In [65]:
# Load the persisted DB
persisted_vectordb_location = project_root/"Database"/"policy_docs"
#Create a Colelction Name
collection_name = 'policy_docs'
# Load the persisted DB
vector_store = Chroma(
    collection_name=collection_name,
    persist_directory=str(persisted_vectordb_location),
    embedding_function=embedding_model

)

In [66]:
qna_user_message_template = """
###Context
Here are some documents and their source that may be relevant to the question mentioned below.
{context}

###Question
{question}
"""

In [67]:
retriever = vector_store.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5}
)

In [68]:
REGISTRY = {
    "support_agent": {
        "v1": {
            "prompt_path": str(Path(r"prompts\rag_agent\v1.txt")).replace("\\","/"),
            "config": {
                "model": "llama-3.1-8b-instant",
                "temperature": 0.0,
                "max_tokens": 512,
            },
            "tools": ["lookup_order"],  # Tool names this agent may call
            "max_tool_steps": 3,

}

    }}

config_loc=project_root/"configs"/"rag_agent.json"

with open(config_loc,"w") as file:
    json.dump(REGISTRY,file,indent=4)


In [69]:

with open(config_loc,"r") as f:
    data=json.load(f)
relitive=data["support_agent"]["v1"]["prompt_path"]
path=project_root/relitive
with open(path,"r") as f:
    qna_system_message=f.read()
model=data["support_agent"]["v1"]["config"]["model"]


In [70]:
@tool
def rag(user_input: str) -> str:

    """
      Generates a response based on the user's input by retrieving relevant documents and using them as context for a GPT-4o model.

      Args:
          user_input (str): The input question or query from the user.

      Returns:
          response (str): Return the generated response or an error message if an exception occurs.

    """

    relevant_document_chunks = retriever.invoke(user_input)
    context_list = [d.page_content + "\n ###Source: " + d.metadata['source'] + "\n\n " for d in relevant_document_chunks]

    context_for_query = ". ".join(context_list)
    # print("context: ", context_for_query)  # Use this to understand what context is provided BTS and to debug.
    prompt = [
        {'role':'system', 'content': qna_system_message},
        {'role': 'user', 'content': qna_user_message_template.format(
            context=context_for_query,
            question=user_input
            )
        }
    ]

    try:
        response = client.chat.completions.create(
        model=model,
        messages=prompt
        )

        prediction = response.choices[0].message.content
    except Exception as e:
        prediction = f'Sorry, I encountered the following error: \n {e}'

    return prediction

In [71]:
@retry(
    stop=stop_after_attempt(4),
    wait=wait_exponential(multiplier=1,min=1,max=10),
    before_sleep=before_sleep_log(logger,logging.WARNING)
)
def rag_tool(user_input):
    answer=rag.invoke(user_input)
    return answer


In [82]:
response=rag_tool("What is the ram on the mobile phone?")
print(response)

4GB of RAM.

Source:
product_Smartphone_description.txt


In [81]:
response=rag_tool("list all keboards available for sale ?")
print(response)

Answer:
We have the following keyboards available for sale:
- Keyboard (Product ID: 5, Price: $261.42) 

Source:
product_Keyboard_description.txt
